# Use your own areas and survey

> Turn boundary and survey files into a state that is ready to rank

::: {.callout-note appearance="simple" icon=false}
**How-to guide: get one job done.** It assumes that you know the basics. New to trufl? Take the [tutorial](../tutorials/01_getting_started.ipynb) first. [About the four kinds of page](../explanation/06_documentation.ipynb).
:::

**You need:** a file of area boundaries with a unique code for each area, and a raster of the survey to use as prior.

In [ ]:
#| hide
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import geopandas as gpd
from fastcore.test import test_eq

from trufl.utils import gridder
from trufl.callbacks import State, MaxCB, MinCB, StdCB, CountCB, PriorCB
from trufl.optimizer import Optimizer
from trufl.sampler import Sampler, rank_to_sample
from trufl.collector import DataCollector

fname_raster = '../files/ground-truth-01-4326-simulated.tif'
areas = gridder(fname_raster)
no_samples = gpd.GeoDataFrame(index=pd.Index([], name='loc_id'), geometry=None, data={'value': None})
prior_state = State(no_samples, areas, cbs=[PriorCB(fname_raster)])()
first_ranks = Optimizer(prior_state).get_rank(is_benefit_x=[True], w_vector=[1], s_method='CP')
first_n = rank_to_sample(first_ranks['rank'].reindex(areas.index).to_numpy(), budget=600, min=1, policy='quantiles')
measurements = DataCollector(fname_raster).collect(
    Sampler(areas).sample(first_n, method='uniform', rng=np.random.default_rng(0)))
state = State(measurements, areas, cbs=[MaxCB(), MinCB(), StdCB(), CountCB(), PriorCB(fname_raster)])()
ranks = Optimizer(state).get_rank(is_benefit_x=[True, True, True, False, True],
                                  w_vector=[0.25, 0.125, 0.125, 0.25, 0.25], n_method='LINEAR1', s_method='CP')

import tempfile
from pathlib import Path
import rasterio

tmp = Path(tempfile.mkdtemp())
boundaries_file, survey_file = tmp/'boundaries.geojson', tmp/'survey.tif'
areas.reset_index().assign(code=lambda d: 'M' + d['loc_id'].astype(str).str.zfill(3))[['code', 'geometry']].to_file(boundaries_file, driver='GeoJSON')
with rasterio.open(fname_raster) as src:
    values, meta = src.read(1), src.meta.copy()
values[:20, :60] = -9999    # a hole in the survey
meta.update(nodata=-9999)
with rasterio.open(survey_file, 'w', **meta) as dst: dst.write(values, 1)

## 1. Read the boundaries

Use your own path for `boundaries_file`, and the name of your code column for `code`.

In [ ]:
import geopandas as gpd

areas = gpd.read_file(boundaries_file).rename(columns={'code': 'loc_id'}).set_index('loc_id')[['geometry']]
assert areas.index.is_unique, 'two areas have the same code'
len(areas)

100

## 2. Match the survey's coordinate system

In [ ]:
import rasterio
from shapely.geometry import box

with rasterio.open(survey_file) as survey:
    areas = areas.to_crs(survey.crs.to_string())
    covered = areas.intersects(box(*survey.bounds))

areas = areas[covered]
print((~covered).sum(), 'areas lie outside the survey')

0 areas lie outside the survey


If that is not 0, find another prior for those areas or plan them separately.

## 3. Compute the state for a first round

In [ ]:
import pandas as pd
from trufl.callbacks import State, PriorCB

no_samples = gpd.GeoDataFrame(index=pd.Index([], name='loc_id'), geometry=None, data={'value': None})
state = State(no_samples, areas, cbs=[PriorCB(survey_file)])().astype(float)
state.head()

,Prior
loc_id,
M000,0.108520
M001,0.133123
M002,0.161802
M003,0.195281
M004,0.213253


Keep the `.astype(float)`. Without it, `isna` does not see the holes in a survey.

## 4. Find and settle the gaps

In [ ]:
gaps = state.index[state['Prior'].isna()]
print(len(gaps), 'areas have no prior:', list(gaps))

2 areas have no prior: ['M009', 'M019']


In [ ]:
#| hide
test_eq(len(gaps) > 0, True)

An area with a gap would [rank as most urgent](../explanation/02_ranking.ipynb#why-a-gap-counts-as-urgent). For each one, either:

- **leave it**, if you do want to visit places that you know nothing about first;
- **fill it** with a value that you can defend, as below;
- **remove it** from `areas` and `state`, and plan it separately.

In [ ]:
state['Prior'] = state['Prior'].fillna(state['Prior'].median())

Next: [set your priorities](03_set_priorities.ipynb).